## NJT Ridership Data Loading & Cleaning (2019)

This section loads the 2019 New Jersey Transit (NJT) ridership data and prepares it for analysis by cleaning, standardizing, and enhancing it with GTFS station information.

---

### Step-by-Step Breakdown

#### 1. **Load Raw Excel Ridership Data**
- Reads from `Sheet2` of `NJT 2019 Ridership.xlsx`, specifically columns B to E.
- Columns renamed to:
  - `Ridership Rank`
  - `Station Name`
  - `Line`
  - `Ridership`
- Drops the first row (header repeated inside data), and ensures correct data types:
  - `Ridership` → float
  - `Ridership Rank` → int
  - Strings are stripped of whitespace.

#### 2. **Group Station Lines**
- Gladstone branch stations reassigned to line `'GLAD'`
- Montclair-Boonton extension stations reassigned to `'M&E'`
- Atlantic City Line (`'ACL'`), `'Hub'`, and outlier station `'Suffern (MLBC)'` are removed
- Replaces "Coast" with "NJCL" and updates station names accordingly

#### 3. **Add Midtown Direct Flag**
- New column: `Midtown Direct Binary`
  - 1 if the station is on `['M&E', 'M-B', 'NEC', 'NJCL']` (indicating Midtown Direct access)
  - 0 otherwise
- Manually sets 0 for known stations lacking Midtown Direct service even if on qualifying lines

#### 4. **Harmonize Station Names**
- Fixes inconsistencies between Excel names and official GTFS `stops.txt` names using a `replace()` mapping
- Example fixes:
  - `'Edison (NEC)'` → `'Edison Station (NEC)'`
  - `'Newark Airport (NEC)'` → `'NEWARK AIRPORT RAILROAD STATION (NEC)'`
- Applies a regex to remove any parentheses content for merging:
  - e.g. `'Bay Head (NJCL)'` → `'Bay Head'`

#### 5. **Merge with GTFS Stops**
- Reads `stops.txt` and extracts `stop_name` for each station
- Creates a normalized lowercase name in both datasets
- Merges the GTFS stop info into the main `Data` dataframe
- Drops unnecessary columns and renames `stop_name` to `Official NJT Station Name`

---

### Final Output

The cleaned DataFrame `Data` now contains:
- Ridership and station attributes
- Consistent line classification
- A Midtown Direct binary indicator
- Matched GTFS station name for downstream merges

In [9]:
import pandas as pd, re, googlemaps, pytz, overpy, math, requests
from datetime import datetime
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup

#API Keys
from config import GOOGLE_MAPS_API_KEY, GOOGLE_API_KEY,CSE_ID

In [10]:
import pandas as pd
import re

Data = pd.read_excel("C:/Users/user00/Downloads/NJT 2019 Ridership.xlsx", sheet_name="Sheet2", usecols="B:E") #CHANGE TO YOUR FILE LOCATION
Data.columns = ["Ridership Rank", "Station Name", "Line", "Ridership"]
Data = Data.iloc[1:] #2nd row downwards

Data['Ridership'] = Data['Ridership'].astype(float)
Data['Ridership Rank'] = Data['Ridership Rank'].astype(int)
Data['Station Name'] = Data['Station Name'].str.strip()
Data['Line'] = Data['Line'].str.strip()
Data.head(10)

GLAD_Stations = ['Gladstone (M&E)', 'Peapack (M&E)', 'Far Hills (M&E)', 'Bernardsville (M&E)', 'Basking Ridge (M&E)', 'Lyons (M&E)', 'Millington (M&E)', 'Gillette (M&E)', 'Stirling (M&E)', 'Murray Hill (M&E)', 'Berkeley Heights (M&E)', 'New Providence (M&E)', 'Murray Hill (M&E)']
ME_Extension = ['Mount Arlington (M-B)', 'Lake Hopatcong (M-B)', 'Netcong (M-B)', 'Mount Olive (M-B)', 'Hackettstown (M-B)']

Data.loc[Data['Station Name'].isin(GLAD_Stations), 'Line'] = 'GLAD'
Data.loc[Data['Station Name'].isin(ME_Extension), 'Line'] = 'M&E'

Data['Station Name'] = Data['Station Name'].replace(ME_Extension, [name.replace('(M-B)', '(M&E)') for name in ME_Extension])
Data['Station Name'] = Data['Station Name'].replace(GLAD_Stations, [name.replace('(M&E)', '(GLAD)') for name in GLAD_Stations])

Data = Data[Data['Line'] != 'ACL']
Data = Data[Data['Line'] != 'Hub']
Data = Data[Data['Station Name'] != 'Suffern (MLBC)']

Data['Line'] = Data['Line'].replace('Coast', 'NJCL')
Data['Station Name'] = Data['Station Name'].str.replace(r'\(Coast\)', '(NJCL)', regex=True)

MidtownDirect_Lines = ['M&E', 'M-B', 'NEC', 'NJCL']
Data['Midtown Direct Binary'] = Data['Line'].apply(lambda x: 1 if x in MidtownDirect_Lines else 0)
stations_to_set_zero = ['East Orange (M&E)', 'Highland Avenue (M&E)', 'Mountain Station (M&E)', 'Elberon (NJCL)', 'Allenhurst (NJCL)', 'Asbury Park (NJCL)', 'Bradley Beach (NJCL)', 'Belmar (NJCL)', 'Spring Lake (CNJCL)', 'Manasquan (NJCL)', 'Point Pleasant Beach (NJCL)', 'Bay Head (NJCL)', 'Mount Arlington (M-B)', 'Lake Hopatcong (M-B)', 'Netcong (M-B)', 'Mount Olive (M-B)', 'Hackettstown (M-B)', 'Little Falls (M-B)', 'WAYNE/ROUTE 23 TRANSIT CENTER [RR]', 'Mountain Lakes (M-B)', 'Lincoln Park (M-B)', 'Boonton (M-B)', 'Towaco (M-B)', 'Mountain View (M-B)']
Data.loc[Data['Station Name'].isin(stations_to_set_zero), 'Midtown Direct Binary'] = 0

### Edit Names to Match Offical NJT Station Names

stop_information = pd.read_csv("C:/Users/user00/Desktop/rail_data/stops.txt") #CHANGE TO YOUR FILE LOCATION

Data['Station Name'] = Data['Station Name'].replace(
    {'Edison (NEC)': 'Edison Station (NEC)',
     'Glen Rock (MLBC)': 'Glen Rock Main Line (MLBC)',
     'Ho-Ho-Kus (MLBC)': 'Hohokus (MLBC)',
     'Jersey Avenue (NEC)': 'Jersey Ave. (NEC)',
     'Middletown (NJCL)': 'MIDDLETOWN NJ (NJCL)',
     'Montclair State Univ. (M-B)': 'MSU (M-B)',
     'Newark Airport (NEC)': 'NEWARK AIRPORT RAILROAD STATION (NEC)',
     'Point Pleasant (NJCL)': 'POINT PLEASANT (NJCL)',
     'Princeton Jct (NEC)': 'PRINCETON JCT. (NEC)',
     'Ramsey, Main Street (MLBC)': 'RAMSEY (MLBC)',
     'Ramsey, Route 17 (MLBC)': 'RAMSEY ROUTE 17 STATION (MLBC)',
     'Trenton (NEC)': 'TRENTON TRANSIT CENTER (NEC)',
     'Watsessing (M-B)': 'WATSESSING AVENUE (M-B)',
     'Wayne, Route 23 (M-B)': 'WAYNE/ROUTE 23 TRANSIT CENTER [RR]',
     'Newark Broad Street (M&E)': 'Newark Broad ST',
     'Middletown (NJCL)': 'Middletown NJ'
     })


def remove_parentheses(text):
    return re.sub(r'\s*\(.*?\)\s*', '', text)


# Clean the station names to ensure they match
stop_information['Station Name Transformed'] = stop_information['stop_name'].str.strip().str.lower()
Data['Station Name Transformed'] = Data['Station Name'].apply(remove_parentheses).str.strip().str.lower()

Data = pd.merge(Data, stop_information, on='Station Name Transformed', how='left')
Data.drop(columns=['zone_id', 'stop_desc', 'stop_code', 'Station Name Transformed'], inplace=True)
Data.rename(columns={'stop_name': 'Official NJT Station Name'}, inplace=True)

## Stop Times Processing and Frequency Feature Engineering

This section processes GTFS stop time data to compute a feature measuring average train frequency per station during service hours. The steps include cleaning the time format, filtering relevant service hours, and aggregating frequencies.

---

### Step-by-Step Breakdown

#### 1. Load GTFS `stop_times.txt`
- Reads in raw GTFS stop time data using `pandas.read_csv`.

#### 2. Fix Invalid Time Format
- GTFS times can exceed 24:00 (e.g., "25:12:00") to represent trips after midnight.
- A helper function `fix_time_format` converts hours `>= 24` by subtracting 24, turning e.g., "25:12:00" into "01:12:00".

#### 3. Convert to Datetime
- Converts the `arrival_time` and `departure_time` columns into proper `datetime` objects using the `%H:%M:%S` format.
- `errors='coerce'` ensures invalid formats are set to NaT.

#### 4. Filter by Service Window
- Retains records where the train's `departure_time` is between 4:00 AM and 2:00 AM.
- Accounts for early morning and late night trains.

#### 5. Extract Hourly Information
- Adds a new column `hour` by extracting the hour of departure.

#### 6. Compute Train Frequency per Hour
- Groups data by `stop_id` and `hour`, then counts the number of train departures (`trains_per_hour`) per station-hour combination.

#### 7. Average Frequency per Station
- Aggregates the hourly frequencies to get the average number of trains per hour (`avg_trains_per_hour`) for each station (`stop_id`).

#### 8. Merge Back to Master Dataset
- Joins the `avg_trains_per_hour` metric into the main `Data` DataFrame by matching on `stop_id`.

---

### Final Output

The main `Data` DataFrame is now enriched with an `avg_trains_per_hour` column representing the typical train frequency at each station during service hours.

In [11]:
stop_times = pd.read_csv('C:/Users/user00/Desktop/rail_data/stop_times.txt') #CHANGE TO YOUR FILE LOCATION

def fix_time_format(time_str):
    if isinstance(time_str, str):  # Ensure the input is a string
        hours, minutes, seconds = map(int, time_str.split(':'))
        if hours >= 24:
            hours -= 24
            return f'{hours:02}:{minutes:02}:{seconds:02}'
    return time_str

# Apply the function to fix the time format
stop_times['arrival_time'] = stop_times['arrival_time'].apply(fix_time_format)
stop_times['departure_time'] = stop_times['departure_time'].apply(fix_time_format)

# Convert the times to datetime objects
stop_times['arrival_time'] = pd.to_datetime(stop_times['arrival_time'], format='%H:%M:%S', errors='coerce')
stop_times['departure_time'] = pd.to_datetime(stop_times['departure_time'], format='%H:%M:%S', errors='coerce')

# Filter the data to include only times between 4:00 am and 2:00 am
filtered_df = stop_times[(stop_times['departure_time'].dt.time >= pd.to_datetime('04:00:00').time()) |
                         (stop_times['departure_time'].dt.time <= pd.to_datetime('02:00:00').time())]

# Create an 'hour' column
filtered_df['hour'] = filtered_df['departure_time'].dt.hour

# Calculate the frequency (trains per hour) per station
frequency_per_station = filtered_df.groupby(['stop_id', 'hour']).size().reset_index(name='trains_per_hour')

# Calculate the average trains per hour for each station
average_frequency_per_station = frequency_per_station.groupby('stop_id')['trains_per_hour'].mean().reset_index(name='avg_trains_per_hour')

# Display the result
Data = pd.merge(Data, average_frequency_per_station, how='left', on='stop_id')

C:\Users\user00\AppData\Local\Temp\ipykernel_62304\3365269255.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['hour'] = filtered_df['departure_time'].dt.hour


## Compute Distance, Duration, and Coordinates Using Google Maps API

This section uses the Google Maps API to enhance each station with:
- **Latitude & Longitude** via the Geocoding API
- **Transit Distance and Duration** to New York Penn Station via the Directions API

---

### Step-by-Step Breakdown

#### 1. API Setup
- A valid Google Maps API key is passed to the `googlemaps.Client`.
- The departure time is set to **7:00 AM on Sept 5, 2025**, in the Eastern Time Zone.

#### 2. Helper: `clean_station_name()`
- Cleans raw station names by removing parentheses and appending `"Station"` if necessary.
- Example: `"Edison (NEC)" → "Edison Station"`

#### 3. Main Function: `get_distance_duration_and_coords()`
- Input: A station name.
- Output: `(distance, duration, lat, lon)` tuple representing:
  - **Distance** to NY Penn Station (e.g., `"24.1 mi"`)
  - **Duration** of train journey including layovers (e.g., `"55 mins"`)
  - **Latitude and Longitude** of the station (via geocoding)

##### a. Geocoding API
- Queries Google Maps to get geographic coordinates for each station.

##### b. Directions API
- Requests transit directions from the origin station to NY Penn.
- Filters only **HEAVY_RAIL** segments (ignores buses).
- Computes:
  - Travel time (`duration['value']`)
  - Layover time between rail segments

#### 4. Apply to All Stations
- Iterates through each station in the `Data` DataFrame.
- Calls `get_distance_duration_and_coords()` per station.
- Collects:
  - Station name
  - Latitude & longitude
  - Distance & duration to NY Penn

#### 5. Create Output DataFrame
- Results are stored in `distance_df`, which includes:
  - `'Station Name'`
  - `'Latitude'`, `'Longitude'`
  - `'Distance from NY Penn (miles)'`
  - `'Duration to NY Penn'`

---

### Final Output

A new `distance_df` DataFrame with geospatial and travel attributes for every NJ Transit station.

In [12]:
import googlemaps
import pytz
import re
from datetime import datetime
import pandas as pd  # for DataFrame

# Replace with your actual Google Maps API key
API_KEY = GOOGLE_MAPS_API_KEY
gmaps = googlemaps.Client(key=API_KEY)

# Define timezone and departure timestamp
eastern = pytz.timezone('America/New_York')
departure_date = datetime(2025, 9, 5, tzinfo=eastern)
departure_time = departure_date.replace(hour=7, minute=0, second=0, microsecond=0)
departure_ts = int(departure_time.timestamp())

def clean_station_name(station_name):
    # Remove parentheses content and ensure single "Station"
    base = re.sub(r'\s*\(.*?\)\s*', '', station_name).strip()
    return base if base.lower().endswith("station") else base + " Station"

def get_distance_duration_and_coords(origin):
    """Return (distance, duration, lat, lon) for a given station."""
    origin_clean = clean_station_name(origin)
    origin_full = f"{origin_clean}, NJ"

    # 1) Geocode to get latitude/longitude
    geocode_result = gmaps.geocode(origin_full)
    if geocode_result:
        loc = geocode_result[0]['geometry']['location']
        lat, lon = loc['lat'], loc['lng']
    else:
        lat = lon = None
        print(f"No geocode for {origin_full}")

    # 2) Request transit directions
    directions = gmaps.directions(
        origin_full,
        "New York Penn Station, New York, NY",
        mode="transit",
        transit_mode="train",
        departure_time=departure_ts
    )
    if not directions:
        print(f"No directions for {origin_full}")
        return None, None, lat, lon

    leg = directions[0]['legs'][0]
    distance = leg['distance']['text']

    # Calculate travel + layover time
    total_train = total_layover = 0
    prev_arrival = None
    for step in leg['steps']:
        details = step.get('transit_details')
        if step['travel_mode'] == 'TRANSIT' and details and details['line']['vehicle']['type'] == 'HEAVY_RAIL':
            secs = step['duration']['value']
            total_train += secs

            dep_ts = details['departure_time']['value']
            arr_ts = details['arrival_time']['value']
            dep = datetime.fromtimestamp(dep_ts, tz=eastern)
            arr = datetime.fromtimestamp(arr_ts, tz=eastern)
            if prev_arrival:
                total_layover += (dep - prev_arrival).total_seconds()
            prev_arrival = arr

    minutes = int((total_train + total_layover) / 60)
    duration = f"{minutes} mins"

    return distance, duration, lat, lon

# Assuming `Data` is your DataFrame with 'Station Name'
results = []
for _, row in Data.iterrows():
    station = row['Station Name']
    dist, dur, lat, lon = get_distance_duration_and_coords(station)
    if dist and dur:
        results.append({
            'Station Name': station,
            'Latitude': lat,
            'Longitude': lon,
            'Distance from NY Penn (miles)': dist,
            'Duration to NY Penn': dur
        })

distance_df = pd.DataFrame(results)

distance_df['Distance from NY Penn (miles)'] = distance_df['Distance from NY Penn (miles)'].str.replace(' mi', '').astype(float)
distance_df['Distance from NY Penn (km)'] = distance_df['Distance from NY Penn (miles)'] * 1.60934
distance_df.drop(columns=['Distance from NY Penn (miles)'], inplace=True)
distance_df['Duration to NY Penn'] = distance_df['Duration to NY Penn'].str.replace(' mins', '').astype(float)
distance_df.rename(columns={'Duration to NY Penn':'Duration to NY Penn (mins)'}, inplace=True)
distance_df.drop(columns=['Latitude','Longitude'], inplace=True) #We have this from stops.txt already

print(distance_df.head(10))

                            Station Name  Duration to NY Penn (mins)  \
0                        Metropark (NEC)                        40.0   
1                   PRINCETON JCT. (NEC)                        55.0   
2                         Hamilton (NEC)                        63.0   
3                    New Brunswick (NEC)                        57.0   
4  NEWARK AIRPORT RAILROAD STATION (NEC)                        29.0   
5                     South Orange (M&E)                        35.0   
6           TRENTON TRANSIT CENTER (NEC)                        70.0   
7                           Summit (M&E)                        41.0   
8                         Metuchen (NEC)                        46.0   
9                        Maplewood (M&E)                        39.0   

   Distance from NY Penn (km)  
0                   39.750698  
1                   78.535792  
2                   87.709030  
3                   52.786352  
4                   20.438618  
5              

In [13]:
Data = pd.merge(distance_df, Data, on = "Station Name")

## Estimating Walk Scores Using OpenStreetMap Amenities

This section estimates a **Walk Score** for each NJ Transit station by analyzing the density and diversity of nearby amenities using **OpenStreetMap** data via the Overpass API.

---

### Step-by-Step Breakdown

#### 1. Setup
- Imports the `overpy` package to interface with the Overpass API.
- Imports `math` for geometric calculations.

#### 2. Function: `query_amenities(lat, lon, radius)`
- Constructs an Overpass QL query to retrieve **nodes tagged with "amenity"** within a circular radius of the station's coordinates.
- Radius is specified in **meters** (e.g., 800m ≈ 10-minute walk).
- Returns all amenity nodes.

#### 3. Function: `calculate_walk_score(amenities, radius)`
- Assigns a **custom weight** to each amenity type:
  - e.g., `park = 5`, `restaurant = 3`, `bus_stop = 1`, etc.
- Computes a **raw score** by summing the weights of matching amenity types.
- Computes the **amenity density** per square kilometer and applies a capped contribution to prevent inflated scores in urban areas.
- Combines the weighted amenity score with capped density for a composite score.

#### 4. Function: `normalize_score(score, max_score)`
- Rescales the composite score to a **0–100 scale**, assuming a maximum achievable value (`max_score = 150` by default).
- Ensures interpretability across stations.

---

### Loop: Compute Walk Score for All Stations

- Iterates over every station in the `Data` DataFrame.
- Retrieves each station’s `Latitude` and `Longitude`.
- Executes:
  - `query_amenities()` to fetch OSM data
  - `calculate_walk_score()` to compute raw score
  - `normalize_score()` to rescale to 0–100
- Stores result in the `Walk_Score` column of the `distance_df` DataFrame.

---

### Final Output

An updated `distance_df` DataFrame now includes a new column:
- `'Walk_Score'`: A numeric estimate of pedestrian-accessible amenity richness, scaled from 0 to 100.
# Walk Ability

In [14]:
import overpy
import math

# Initialize the Overpass API
api = overpy.Overpass()

# Function to query amenities from OpenStreetMap using Overpass API
def query_amenities(lat, lon, radius):
    # Overpass query to get amenities within a radius of the given coordinates
    query = f"""
    [out:json];
    (
      node["amenity"](around:{radius},{lat},{lon});
    );
    out body;
    """
    result = api.query(query)
    return result.nodes

# Function to calculate walk score based on amenities and density
def calculate_walk_score(amenities, radius):
    # Define weights for different types of amenities (you can customize this)
    amenity_weights = {
        'cafe': 2,
        'restaurant': 3,
        'school': 4,
        'park': 5,
        'hospital': 4,
        'supermarket': 3,
        'bus_stop': 1,
        'bank': 2,
    }

    score = 0
    # Calculate score based on the type and count of amenities
    for node in amenities:
        amenity_type = node.tags.get("amenity")
        if amenity_type in amenity_weights:
            score += amenity_weights[amenity_type]

    # Calculate density of amenities
    area = math.pi * (radius / 1000) ** 2  # Convert radius to km for area
    density = len(amenities) / area if area > 0 else 0

    # Cap the effect of density to avoid inflated scores (adjust multiplier as needed)
    density_cap = min(density, 10)  # Cap the density contribution

    # Combine score with capped density contribution
    weighted_score = score + (density_cap * 2)  # Adjust weighting for density here

    return weighted_score

# Function to normalize the score to a scale of 0-100
def normalize_score(score, max_score):
    # Assuming the realistic max score based on previous tests (adjust max_score)
    adjusted_max_score = max_score if max_score else 150  # Use 150 as default if not specified
    return min(100, (score / adjusted_max_score) * 100)

# Define a radius for walking distance (e.g., 800 meters ~ 10 minutes walk)
radius = 800
max_score = 100  # Set a realistic max score for normalization (adjust after some testing)

# Loop through each station in the DataFrame and calculate walk scores
Data['Walk_Score'] = 0  # Initialize a column for walk scores

for index, row in Data.iterrows():
    lat = row['stop_lat']
    lon = row['stop_lon']

    # Query amenities within the radius
    amenities = query_amenities(lat, lon, radius)

    # Calculate walk score including density
    raw_score = calculate_walk_score(amenities, radius)

    # Normalize the score to a scale of 0-100
    walk_score = normalize_score(raw_score, max_score)

    # Store the walk score in the DataFrame
    Data.at[index, 'Walk_Score'] = walk_score

# Print the updated DataFrame with walk scores
print(Data[['Station Name', 'Walk_Score']])

C:\Users\user00\AppData\Local\Temp\ipykernel_62304\1521486435.py:80: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '13.92605752054084' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  Data.at[index, 'Walk_Score'] = walk_score


                              Station Name  Walk_Score
0                          Metropark (NEC)   32.000000
1                     PRINCETON JCT. (NEC)   37.000000
2                           Hamilton (NEC)   13.926058
3                      New Brunswick (NEC)  100.000000
4    NEWARK AIRPORT RAILROAD STATION (NEC)    2.984155
..                                     ...         ...
133                         Peapack (GLAD)    2.984155
134                      Mount Tabor (M&E)    8.973592
135                          Lebanon (RVL)    6.984155
136                   Mountain Lakes (M-B)    0.994718
137                      Mount Olive (M&E)    6.963029

[138 rows x 2 columns]


## Parking Capacity Scraping with Google Custom Search and BeautifulSoup

This section retrieves **station-level parking capacity** from the NJ Transit website using a combination of:

- **Google Custom Search API** to locate official NJ Transit station webpages
- **BeautifulSoup** to scrape the number of standard parking spaces
- **RobotFileParser** to ensure scraping complies with `robots.txt`

### Step-by-Step Breakdown

1. **Check Robots.txt Permissions**
   We use Python’s `RobotFileParser` to download and parse the `robots.txt` file from `njtransit.com`. This ensures that scraping is permitted for the desired URLs.

2. **Search Station URLs via Google Custom Search API**
   The function `search_station_njt()` queries the Google Custom Search API to find the NJ Transit webpage for each station name.
   - It narrows the search to results from `site:njtransit.com/station`
   - Filters out irrelevant results like `light-rail`, `bus-terminal`, or `elevators`

3. **Scrape Parking Information from Station Pages**
   For each station URL returned by the API:
   - The function `get_parking_info()` fetches the HTML content
   - Parses it using BeautifulSoup
   - Searches for `<p>` tags containing "Standard Spaces:"
   - Extracts and sums the number of parking spaces

4. **Aggregate Parking Data for All Stations**
   The main function `get_parking_for_stations()`:
   - Iterates through each station in the `Data` DataFrame
   - Searches for the correct station URL
   - Scrapes the total number of parking spaces
   - Appends the results to a new DataFrame (`parking_df`)

5. **Merge Parking Data with Main Dataset**
   After collecting the parking information, we merge `parking_df` with the original `Data` DataFrame using `'Official NJT Station Name'` as the key. This integrates the `Total Parking Spaces` as a feature for further analysis and modeling.

In [15]:
import requests
from urllib.robotparser import RobotFileParser

# Define the base URL
base_url = "https://www.njtransit.com/"

# Fetch the robots.txt content
robots_url = base_url + "robots.txt"
response = requests.get(robots_url)
response.raise_for_status()

# Initialize and load the parser
parser = RobotFileParser()
parser.parse(response.text.splitlines())

# Check if scraping is allowed for the base URL
is_allowed = parser.can_fetch("*", base_url)

print(f"Scraping allowed: {is_allowed}")


Scraping allowed: True


In [16]:
from bs4 import BeautifulSoup
import requests

# Function to search for a station on NJT's website
def search_station_njt(station_name):
    search_url = f"https://www.googleapis.com/customsearch/v1"
    # Modify the query to include 'rail station' to emphasize rail-related pages
    params = {
        'key': GOOGLE_API_KEY,
        'cx': CSE_ID,  # Custom Search Engine ID
        'q': f"site:njtransit.com/station {station_name} rail station"
    }

    response = requests.get(search_url, params=params)
    if response.status_code == 200:
        search_results = response.json()
        for item in search_results.get('items', []):
            link = item['link']
            # Exclude URLs with "light-rail", "elevators", "bus-terminal", "park-ride" in the URL
            if '/station/' in link and all(excluded not in link for excluded in ['light-rail', 'elevators', 'bus-terminal', 'park-ride']):
                return link
    return None

# Function to scrape parking information from a station page
def get_parking_info(station_url):
    response = requests.get(station_url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        parking_section = soup.find_all('p', class_='card-text mb-0 px-gutter py-1 bg-light')

        total_parking_spaces = 0

        for parking_info in parking_section:
            if 'Standard Spaces:' in parking_info.text:
                parking_capacity = parking_info.text.strip().replace('Standard Spaces:', '').strip()
                total_parking_spaces += int(parking_capacity)

        return total_parking_spaces
    else:
        print(f"Failed to retrieve the page for {station_url}. Status code: {response.status_code}")
        return 0

# Function to get parking information for all stations and return a DataFrame
def get_parking_for_stations(df):
    station_names = []
    parking_spaces = []

    for index, row in df.iterrows():
        station_name = row['Official NJT Station Name']
        print(f"Searching for {station_name}...")

        # Search for the station on NJT's website
        station_url = search_station_njt(station_name)
        if station_url:
            print(f"Found URL: {station_url}")
            # Scrape the parking information
            total_parking = get_parking_info(station_url)
        else:
            print(f"No URL found for {station_name}.")
            total_parking = 0

        # Store results
        station_names.append(station_name)
        parking_spaces.append(total_parking)

    # Create a new DataFrame with station names and parking spaces
    parking_df = pd.DataFrame({
        'Official NJT Station Name': station_names,
        'Total Parking Spaces': parking_spaces
    })

    return parking_df

# Apply the function to get the parking data
parking_data = get_parking_for_stations(Data)

Searching for METROPARK...
Found URL: https://www.njtransit.com/station/metropark-station
Searching for PRINCETON JCT....
Found URL: https://www.njtransit.com/station/princeton-junction-station
Searching for HAMILTON...
Found URL: https://www.njtransit.com/station/hamilton-station
Searching for NEW BRUNSWICK...
Found URL: https://www.njtransit.com/station/new-brunswick-station
Searching for NEWARK AIRPORT RAILROAD STATION...
Found URL: https://www.njtransit.com/station/newark-airport-rail-station
Searching for SOUTH ORANGE...
Found URL: https://www.njtransit.com/station/south-orange-station
Searching for TRENTON TRANSIT CENTER...
Found URL: https://www.njtransit.com/station/trenton-transit-center
Searching for SUMMIT...
Found URL: https://www.njtransit.com/station/summit-station
Searching for METUCHEN...
Found URL: https://www.njtransit.com/station/metuchen-station
Searching for MAPLEWOOD...
Found URL: https://www.njtransit.com/station/maplewood-station
Searching for ELIZABETH...
Found

In [17]:
# Values Not Picked Up by CSE Search
parking_data.loc[parking_data['Official NJT Station Name'] == 'BASKING RIDGE', 'Total Parking Spaces'] = 89
parking_data.loc[parking_data['Official NJT Station Name'] == 'STIRLING', 'Total Parking Spaces'] = 39
parking_data.loc[parking_data['Official NJT Station Name'] == 'WESTWOOD', 'Total Parking Spaces'] = 220
parking_data.loc[parking_data['Official NJT Station Name'] == 'GILLETTE', 'Total Parking Spaces'] = 82
parking_data.loc[parking_data['Official NJT Station Name'] == 'PARK RIDGE', 'Total Parking Spaces'] = 132
parking_data.loc[parking_data['Official NJT Station Name'] == 'WAYNE/ROUTE 23 TRANSIT CENTER [RR]', 'Total Parking Spaces'] = 996
parking_data.loc[parking_data['Official NJT Station Name'] == 'ELBERON', 'Total Parking Spaces'] = 222
parking_data.loc[parking_data['Official NJT Station Name'] == 'WESMONT', 'Total Parking Spaces'] = 216
parking_data.loc[parking_data['Official NJT Station Name'] == 'BROADWAY', 'Total Parking Spaces'] = 80
parking_data.loc[parking_data['Official NJT Station Name'] == 'HIGHLAND AVENUE', 'Total Parking Spaces'] = 31
parking_data.loc[parking_data['Official NJT Station Name'] == 'MOUNTAIN STATION', 'Total Parking Spaces'] = 88
parking_data.loc[parking_data['Official NJT Station Name'] == 'BERKELEY HEIGHTS', 'Total Parking Spaces'] = 219
parking_data.loc[parking_data['Official NJT Station Name'] == 'NETHERWOOD', 'Total Parking Spaces'] = 158
parking_data.loc[parking_data['Official NJT Station Name'] == 'NEW PROVIDENCE', 'Total Parking Spaces'] = 114
parking_data.loc[parking_data['Official NJT Station Name'] == 'UPPER MONTCLAIR', 'Total Parking Spaces'] = 109
parking_data.loc[parking_data['Official NJT Station Name'] == 'NORTH ELIZABETH', 'Total Parking Spaces'] = 123
parking_data.loc[parking_data['Official NJT Station Name'] == 'WATCHUNG AVENUE', 'Total Parking Spaces'] = 93
parking_data.loc[parking_data['Official NJT Station Name'] == 'PRINCETON', 'Total Parking Spaces'] = 247
parking_data.loc[parking_data['Official NJT Station Name'] == 'BERNARDSVILLE', 'Total Parking Spaces'] = 143
parking_data.loc[parking_data['Official NJT Station Name'] == 'HIGH BRIDGE', 'Total Parking Spaces'] = 40

In [18]:
Data = pd.merge(Data, parking_data, on='Official NJT Station Name', how='left')

In [19]:
Data.to_csv("Stop_Data.csv",index=False)

In [20]:
Data = pd.read_csv("Stop_Data.csv")

In [21]:
Data

,Station Name,Duration to NY Penn (mins),Distance from NY Penn (km),Ridership Rank,Line,Ridership,Midtown Direct Binary,stop_id,Official NJT Station Name,stop_lat,stop_lon,avg_trains_per_hour,Walk_Score,Total Parking Spaces
0,Metropark (NEC),40.0,39.750698,5,NEC,7537.0,1,83,METROPARK,40.568640,-74.329394,12.227273,32.000000,3491
1,PRINCETON JCT. (NEC),55.0,78.535792,6,NEC,6817.0,1,125,PRINCETON JCT.,40.316316,-74.623753,22.363636,37.000000,4118
2,Hamilton (NEC),63.0,87.709030,7,NEC,5178.0,1,32905,HAMILTON,40.255309,-74.704120,11.727273,13.926058,3539
3,New Brunswick (NEC),57.0,52.786352,8,NEC,4700.0,1,103,NEW BRUNSWICK,40.497278,-74.445751,12.818182,100.000000,4441
4,NEWARK AIRPORT RAILROAD STATION (NEC),29.0,20.438618,9,NEC,4253.0,1,37953,NEWARK AIRPORT RAILROAD STATION,40.704415,-74.190717,21.045455,2.984155,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,Peapack (GLAD),87.0,70.971894,148,GLAD,36.0,0,117,PEAPACK,40.708794,-74.658469,5.681818,2.984155,0
134,Mount Tabor (M&E),73.0,61.798656,149,M&E,30.0,1,94,MOUNT TABOR,40.875904,-74.481915,4.571429,8.973592,0
135,Lebanon (RVL),101.0,82.559142,150,RVL,21.0,0,68,LEBANON,40.636903,-74.835766,1.533333,6.984155,0
136,Mountain Lakes (M-B),90.0,57.936240,151,M-B,17.0,0,96,MOUNTAIN LAKES,40.885947,-74.433604,2.166667,0.994718,0
